In [1]:
!pip install sentence-transformers transformers hnswlib numpy nltk

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 853.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.1 MB/s eta 0:00:00
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp311-cp311-lin

In [12]:
from google.colab import files

# Upload manual do arquivo domcasmurro.txt
uploaded = files.upload()

Saving domcasmurro.txt to domcasmurro (1).txt


In [33]:
import re
import numpy as np
import hnswlib
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import pickle


def read_file(file_path):
    with open(file_path, 'r', encoding='latin-1') as f:
        return f.read()

def extract_chapters(text):

    chapters = re.split(r'CAPÍTULO [IVXLCDM]+', text)[1:]
    chapter_titles = re.findall(r'CAPÍTULO [IVXLCDM]+.*?(?=\n)', text)

    formatted_chapters = []
    for i in range(min(len(chapter_titles), len(chapters))):
        chapter_text = chapters[i].strip()
        formatted_chapters.append(f"{chapter_titles[i]}\n{chapter_text}")

    return formatted_chapters


print("Carregando o texto do livro...")
text = read_file('domcasmurro.txt')

print("Dividindo o texto em capítulos...")
chapters = extract_chapters(text)

print(f"\nTotal de capítulos: {len(chapters)}")

print("\nCarregando modelo de embeddings...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Gerando embeddings para os capítulos...")
embeddings = model.encode(chapters)
print(f"Dimensão dos embeddings: {embeddings.shape}")

print("\nCriando índice de busca vetorial...")
index = hnswlib.Index(space='cosine', dim=embeddings.shape[1])
index.init_index(max_elements=len(embeddings), ef_construction=200, M=16)
index.add_items(embeddings)


Carregando o texto do livro...
Dividindo o texto em capítulos...

Total de capítulos: 147

Carregando modelo de embeddings...
Gerando embeddings para os capítulos...
Dimensão dos embeddings: (147, 384)

Criando índice de busca vetorial...


In [34]:
def retrieve_context(query, top_k=1):
    query_embedding = model.encode([query])

    ids, distances = index.knn_query(query_embedding, k=top_k)

    retrieved_chapters = [chapters[idx] for idx in ids[0]]
    return retrieved_chapters, distances[0]

print("\nCarregando modelos de QA...")
qa_model1 = pipeline("question-answering", model="distilbert-base-uncased-distilled-squad")
qa_model2 = pipeline("question-answering", model="deepset/roberta-base-squad2")

Device set to use cpu



Carregando modelos de QA...


Device set to use cpu


In [35]:
def answer_question(question, model_name="distilbert"):

    retrieved_contexts, distances = retrieve_context(question, top_k=1)
    context = retrieved_contexts[0]

    print(f"\nCapítulo recuperado (similaridade: {1-distances[0]:.4f}):")
    print(context[:300] + "...")

    if model_name == "distilbert":
        qa_model = qa_model1
        print("\nUsando modelo: distilbert-base-uncased-distilled-squad")
    else:
        qa_model = qa_model2
        print("\nUsando modelo: deepset/roberta-base-squad2")

    answer = qa_model(question=question, context=context)

    return answer, context

perguntas = [
    "Quem é Capitu?",
    "Por que Bentinho deveria ir para o seminário?",
    "Quem era José Dias?",
    "O que motivou a construção da casa no Engenho Novo?"
]

resultados = {}

for pergunta in perguntas:
    print("\n" + "="*80)
    print(f"PERGUNTA: {pergunta}")
    print("="*80)

    resposta1, contexto = answer_question(pergunta, "distilbert")
    print(f"\nResposta (distilbert): {resposta1['answer']}")
    print(f"Score: {resposta1['score']:.4f}")

    resposta2, _ = answer_question(pergunta, "roberta")
    print(f"\nResposta (roberta): {resposta2['answer']}")
    print(f"Score: {resposta2['score']:.4f}")

    resultados[pergunta] = {
        "contexto": contexto[:500] + "...",
        "distilbert": {
            "resposta": resposta1['answer'],
            "score": resposta1['score']
        },
        "roberta": {
            "resposta": resposta2['answer'],
            "score": resposta2['score']
        }
    }


PERGUNTA: Quem é Capitu?

Capítulo recuperado (similaridade: 0.6274):
CAPÍTULO CXXXVIII / CAPITU QUE ENTRA 
/ CAPITU QUE ENTRA 
Quando levantei a cabeça, dei com a figura de Capitu diante de mim. Eis aí outro lance, que parecerá de teatro, e é tão natural como o primeiro, uma vez que a mãe e o filho iam à missa, e Capitu não saía sem falar-me. Era já um falar seco e b...

Usando modelo: distilbert-base-uncased-distilled-squad

Resposta (distilbert): e murmurou: 
--Sei a razão disto
Score: 0.0112

Capítulo recuperado (similaridade: 0.6274):
CAPÍTULO CXXXVIII / CAPITU QUE ENTRA 
/ CAPITU QUE ENTRA 
Quando levantei a cabeça, dei com a figura de Capitu diante de mim. Eis aí outro lance, que parecerá de teatro, e é tão natural como o primeiro, uma vez que a mãe e o filho iam à missa, e Capitu não saía sem falar-me. Era já um falar seco e b...

Usando modelo: deepset/roberta-base-squad2

Resposta (roberta): outro lance
Score: 0.0088

PERGUNTA: Por que Bentinho deveria ir para o seminário?



In [36]:
print("\nSistema RAG implementado")

print("\nResumo dos resultados:")
for pergunta, resultado in resultados.items():
    print(f"\nPergunta: {pergunta}")
    print(f"Resposta (distilbert): {resultado['distilbert']['resposta']}")
    print(f"Resposta (roberta): {resultado['roberta']['resposta']}")

    print("\nComparação de scores:")
    print(f"  DistilBERT: {resultado['distilbert']['score']:.4f}")
    print(f"  RoBERTa: {resultado['roberta']['score']:.4f}")

    if resultado['distilbert']['score'] > resultado['roberta']['score']:
        melhor_modelo = "DistilBERT"
        diferenca = resultado['distilbert']['score'] - resultado['roberta']['score']
    else:
        melhor_modelo = "RoBERTa"
        diferenca = resultado['roberta']['score'] - resultado['distilbert']['score']

    print(f"  Melhor modelo: {melhor_modelo} (diferença de {diferenca:.4f})")


Sistema RAG implementado e testado com sucesso!

Resumo dos resultados:

Pergunta: Quem é Capitu?
Resposta (distilbert): e murmurou: 
--Sei a razão disto
Resposta (roberta): outro lance

Comparação de scores:
  DistilBERT: 0.0112
  RoBERTa: 0.0088
  Melhor modelo: DistilBERT (diferença de 0.0023)

Pergunta: Por que Bentinho deveria ir para o seminário?
Resposta (distilbert): não iria
Resposta (roberta): --

Comparação de scores:
  DistilBERT: 0.0107
  RoBERTa: 0.0006
  Melhor modelo: DistilBERT (diferença de 0.0100)

Pergunta: Quem era José Dias?
Resposta (distilbert): Não importa, disse-me José Dias
Resposta (roberta): --

Comparação de scores:
  DistilBERT: 0.0023
  RoBERTa: 0.0065
  Melhor modelo: RoBERTa (diferença de 0.0042)

Pergunta: O que motivou a construção da casa no Engenho Novo?
Resposta (distilbert): que casou há dias com aquela moça
Resposta (roberta): um relógio

Comparação de scores:
  DistilBERT: 0.0000
  RoBERTa: 0.0031
  Melhor modelo: RoBERTa (diferença de 0.0031)